# VULKN AI Service — Week 3 Mini-Project Demo

**Person 1 — ML/AI Engineer**

## What I built this week
A machine learning anomaly detector for HTTP web application traffic.
Given a raw HTTP request, the model classifies it as **normal** or **anomalous (attack)**.

## Dataset
- HTTP CSIC 2010 — 61,065 real HTTP requests from a Spanish research institute
- 36,000 normal requests + 25,065 attack requests (SQL injection, XSS, path traversal, buffer overflow)

## Algorithm
Isolation Forest (unsupervised anomaly detection)
- Trains on normal traffic only — no attack labels needed
- Flags requests that "don't look like" normal traffic

## Outcome
- contamination=0.10 → catches 30.8% of attacks with 68.4% precision
- Saved model: isolation_forest_final.joblib

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import IsolationForest

df = pd.read_csv('datasets/csic_database.csv')

print("=" * 55)
print("STEP 1: DATASET LOADED")
print("=" * 55)
print(f"File        : csic_database.csv")
print(f"Total rows  : {len(df):,} HTTP requests")
print(f"Columns     : {df.shape[1]}")
print()

counts = df['classification'].value_counts()
n_normal = counts[0]
n_attack = counts[1]
print(f"Normal requests  (0): {n_normal:,}  ({n_normal/len(df)*100:.1f}%)")
print(f"Attack requests  (1): {n_attack:,}  ({n_attack/len(df)*100:.1f}%)")
print()
print("Attack types in this dataset:")
print("  SQL Injection, XSS, Path Traversal,")
print("  Buffer Overflow, CSRF, Parameter Tampering")
print()
print("Sample NORMAL request:")
normal_row = df[df['classification']==0].iloc[0]
print(f"  Method : {normal_row['Method']}")
print(f"  URL    : {str(normal_row['URL'])[:70]}")
print()
print("Sample ATTACK request:")
attack_row = df[df['classification']==1].iloc[0]
print(f"  Method : {attack_row['Method']}")
print(f"  URL    : {str(attack_row['URL'])[:70]}")

STEP 1: DATASET LOADED
File        : csic_database.csv
Total rows  : 61,065 HTTP requests
Columns     : 17

Normal requests  (0): 36,000  (59.0%)
Attack requests  (1): 25,065  (41.0%)

Attack types in this dataset:
  SQL Injection, XSS, Path Traversal,
  Buffer Overflow, CSRF, Parameter Tampering

Sample NORMAL request:
  Method : GET
  URL    : http://localhost:8080/tienda1/index.jsp HTTP/1.1

Sample ATTACK request:
  Method : GET
  URL    : http://localhost:8080/tienda1/publico/anadir.jsp?id=2&nombre=Jam%F3n+I


In [2]:
def clean_url(url):
    return re.sub(r'\s+HTTP/\d\.\d\s*$', '', str(url)).strip()

def count_params(url):
    url_str = str(url)
    if '?' in url_str:
        q = url_str.split('?')[1]
        q = re.sub(r'\s+HTTP/\d\.\d\s*$', '', q)
        return len(q.split('&'))
    return 0

df['url_clean']      = df['URL'].apply(clean_url)
df['request_length'] = df['url_clean'].str.len()
df['method_encoded'] = df['Method'].map({'GET':0,'POST':1,'PUT':2}).fillna(3).astype(int)
df['num_parameters'] = df['url_clean'].apply(count_params)
df['body_length']    = df['content'].fillna('').astype(str).str.len()

FEATURES  = ['request_length','method_encoded','num_parameters','body_length']
X_all     = df[FEATURES]
y         = df['classification']
X_normal  = X_all[y == 0]
X_attack  = X_all[y == 1]

print("=" * 55)
print("STEP 2: FEATURE ENGINEERING (Day 12 work)")
print("=" * 55)
print("Converted raw HTTP text into 4 numbers per request:")
print()
print(f"  1. request_length  = length of URL in characters")
print(f"  2. method_encoded  = GET=0, POST=1, PUT=2")
print(f"  3. num_parameters  = number of key=value pairs in URL")
print(f"  4. body_length     = length of request body")
print()
print("Why these 4? Because attacks differ from normal:")
for feat in FEATURES:
    n_mean = X_all[y==0][feat].mean()
    a_mean = X_all[y==1][feat].mean()
    diff   = ((a_mean - n_mean) / n_mean * 100) if n_mean > 0 else 0
    print(f"  {feat:20s}: Normal avg={n_mean:6.1f}  Attack avg={a_mean:6.1f}  ({diff:+.0f}%)")

STEP 2: FEATURE ENGINEERING (Day 12 work)
Converted raw HTTP text into 4 numbers per request:

  1. request_length  = length of URL in characters
  2. method_encoded  = GET=0, POST=1, PUT=2
  3. num_parameters  = number of key=value pairs in URL
  4. body_length     = length of request body

Why these 4? Because attacks differ from normal:
  request_length      : Normal avg=  70.0  Attack avg=  97.6  (+39%)
  method_encoded      : Normal avg=   0.2  Attack avg=   0.4  (+86%)
  num_parameters      : Normal avg=   1.2  Attack avg=   2.2  (+91%)
  body_length         : Normal avg=  20.3  Attack avg=  48.6  (+139%)


In [3]:
print("=" * 55)
print("STEP 3: MODEL TRAINED (Day 13 work)")
print("=" * 55)

model = IsolationForest(n_estimators=100, contamination=0.10, random_state=42)
model.fit(X_normal)

print(f"Algorithm    : Isolation Forest")
print(f"Trained on   : {len(X_normal):,} NORMAL requests only")
print(f"n_estimators : {model.n_estimators} random trees")
print(f"contamination: {model.contamination}")
print(f"Threshold    : {model.offset_:.4f}")
print()
print("How it works:")
print("  The model builds 100 random trees from normal requests.")
print("  Normal requests need MANY cuts to isolate → score closer to 0")
print("  Attack requests need FEW cuts to isolate → score more negative")
print("  Any request with score below threshold → flagged as anomalous")

STEP 3: MODEL TRAINED (Day 13 work)
Algorithm    : Isolation Forest
Trained on   : 36,000 NORMAL requests only
n_estimators : 100 random trees
contamination: 0.1
Threshold    : -0.6004

How it works:
  The model builds 100 random trees from normal requests.
  Normal requests need MANY cuts to isolate → score closer to 0
  Attack requests need FEW cuts to isolate → score more negative
  Any request with score below threshold → flagged as anomalous


In [4]:
sample_normal = X_normal.head(5).copy()
sample_attack  = X_attack.head(5).copy()
sample_normal['true_label'] = 0
sample_attack['true_label'] = 1

samples   = pd.concat([sample_normal, sample_attack]).reset_index(drop=True)
X_samples = samples[FEATURES]
preds     = model.predict(X_samples)
scores    = model.score_samples(X_samples)

method_rev = {0:'GET', 1:'POST', 2:'PUT'}

print("=" * 70)
print("STEP 4: ANOMALY SCORES — 5 NORMAL + 5 ATTACK REQUESTS")
print("=" * 70)
print(f"  {'#':>2}  {'True':8}  {'Score':8}  {'Result':12}  {'Correct?':10}  URL length / params / body")
print("-" * 70)

for i in range(10):
    true   = samples['true_label'].iloc[i]
    pred   = preds[i]
    score  = scores[i]
    feats  = X_samples.iloc[i].tolist()
    method = method_rev.get(int(feats[1]), '?')
    label  = 'NORMAL' if true == 0 else 'ATTACK'
    result = 'anomalous' if pred == -1 else 'normal'
    ok     = '✓ correct' if (pred==-1 and true==1) or (pred==1 and true==0) else '✗ wrong'
    print(f"  {i+1:2d}  {label:8}  {score:8.4f}  {result:12}  {ok:10}  len={int(feats[0])} params={int(feats[2])} body={int(feats[3])}")

print()
print(f"Threshold = {model.offset_:.4f}  (score below this → flagged as anomalous)")
print()
print("EXPLANATION OF SCORES:")
print("  More negative score = more anomalous = further from normal cluster")
print("  Score > threshold → predicted normal")
print("  Score < threshold → predicted anomalous")
print()
print("WHY SOME ATTACKS SHOW AS 'normal':")
print("  Some attacks have short URLs, use GET, 0 params, empty body")
print("  With only 4 features, they look identical to normal requests")
print("  Week 6 adds more features to fix this (response time, error keywords)")

STEP 4: ANOMALY SCORES — 5 NORMAL + 5 ATTACK REQUESTS
   #  True      Score     Result        Correct?    URL length / params / body
----------------------------------------------------------------------
   1  NORMAL     -0.5239  normal        ✓ correct   len=39 params=0 body=0
   2  NORMAL     -0.5684  normal        ✓ correct   len=117 params=5 body=0
   3  NORMAL     -0.5300  normal        ✓ correct   len=48 params=0 body=68
   4  NORMAL     -0.5538  normal        ✓ correct   len=116 params=5 body=0
   5  NORMAL     -0.5756  normal        ✓ correct   len=52 params=0 body=63
   6  ATTACK     -0.6497  anomalous     ✓ correct   len=195 params=5 body=0
   7  ATTACK     -0.5978  normal        ✗ wrong     len=48 params=0 body=146
   8  ATTACK     -0.6424  anomalous     ✓ correct   len=126 params=5 body=0
   9  ATTACK     -0.5667  normal        ✗ wrong     len=48 params=0 body=77
  10  ATTACK     -0.5042  normal        ✗ wrong     len=40 params=0 body=0

Threshold = -0.6004  (score below th

In [5]:
from sklearn.metrics import confusion_matrix

print("=" * 65)
print("STEP 5: CONTAMINATION TUNING RESULTS (Day 14 work)")
print("=" * 65)
print(f"  {'Contamination':15} {'Recall':10} {'Precision':12} {'False Alarms':15} {'Attacks Caught'}")
print("-" * 65)

for c in [0.05, 0.10, 0.20]:
    m = IsolationForest(n_estimators=100, contamination=c, random_state=42)
    m.fit(X_normal)
    preds_c = m.predict(X_all)
    pred_b  = (preds_c == -1).astype(int)
    cm      = confusion_matrix(y, pred_b)
    tp, fn, fp, tn = cm[1][1], cm[1][0], cm[0][1], cm[0][0]
    recall  = tp / (tp + fn) * 100
    prec    = tp / (tp + fp) * 100 if (tp+fp)>0 else 0
    chosen  = " ← CHOSEN" if c == 0.10 else ""
    print(f"  {c:<15} {recall:8.1f}%  {prec:10.1f}%  {fp:13,}  {tp:,}{chosen}")

print()
print("Decision: contamination=0.10 chosen for VULKN")
print("Reason: best balance between catching attacks and avoiding false alarms")

STEP 5: CONTAMINATION TUNING RESULTS (Day 14 work)
  Contamination   Recall     Precision    False Alarms    Attacks Caught
-----------------------------------------------------------------
  0.05                18.4%        74.1%          1,618  4,623
  0.1                 30.8%        68.4%          3,563  7,719 ← CHOSEN
  0.2                 45.0%        64.3%          6,264  11,280

Decision: contamination=0.10 chosen for VULKN
Reason: best balance between catching attacks and avoiding false alarms
